# 摘要记忆

> **用 LLM 生成的滚动摘要压缩较早的对话轮次：用逐字回忆换取无限对话长度。**

在[上一份 notebook](../02_sliding_window_memory/sliding_window_memory.ipynb)中我们看到**滑动窗口记忆**通过完全丢弃旧消息来限制成本。这种硬截断意味着 Agent 甚至*不知道*它遗忘了什么。

**摘要记忆**采用不同的方法。它不是丢弃历史，而是*压缩*历史。想象阅读一本长书并在每章结尾写一页摘要。你不能再逐字逐句引用原书，但你仍然知道发生了什么。一个辅助 LLM 调用会定期将旧消息浓缩为运行中的文本摘要。Agent 失去了精确措辞，但能在任意长的对话中保留大意（关键事实、决策和上下文）。

**代价：** 摘要是有损的（它们无法完美重建原文）。每次压缩循环都可能丢失细节、转移重点或微妙地扭曲事实。经过多次循环后，这种**摘要漂移**会累积。Agent 的"记忆"可能与实际发生的情况产生偏差。

**完成本 notebook 后你将理解：**
- 如何使用 Anthropic SDK 从头构建滚动摘要记忆系统。
- 摘要循环：何时触发、使用什么提示词，以及摘要如何演化。
- 摘要如何在长对话中漂移，附有对照实验和可视化。
- 针对漂移和信息丢失的实用缓解措施。

## 核心概念

- **滚动摘要**：一个单一的文本块，随着对话的增长而增量更新。每次更新将新消息合并到现有摘要中。
- **摘要提示词**：给 LLM 生成摘要的指令。其措辞控制保留什么（事实、决策、语气）和丢弃什么。
- **刷新触发器**：决定*何时*重新摘要的规则。常见选项：每 *n* 条消息后、缓冲区超过 token 阈值时，或每轮都刷新。
- **摘要漂移**：事实在重复摘要周期中逐渐失真。细节会随着时间的推移被软化、合并或完全丢失。
- **压缩比**：与所替代的原始消息相比，摘要缩短了多少。更高的压缩意味着更多的信息丢失。
- **缓冲区**：在摘要旁按原文保留的最近消息。这为 LLM 提供了最新对话内容的精确上下文，而较早的上下文则存在于压缩摘要中。

## 模型准备

In [2]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")
print(type(model)) # <class 'langchain_deepseek.chat_models.ChatDeepSeek'>

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 核心实现

设计分为两部分：

1. **缓冲区**：按原文存储的最近消息（字典列表）。
2. **摘要**：一个压缩所有旧消息的字符串。

当缓冲区达到 `max_buffer_size` 条消息时，我们用*摘要提示词*调用 LLM。该提示词将当前摘要 + 缓冲区合并为一个新摘要。然后我们清空缓冲区。

每轮聊天 LLM 接收：`[system: summary_context] + buffer_messages`。

In [3]:
SUMMARIZER_PROMPT = """你是一个对话摘要器。根据现有摘要和新消息，
生成一个更新后的摘要，要求：
1. 保留所有关键事实（名称、数字、偏好、决策）。
2. 记录任何未解决的问题或待定话题。
3. 保持简洁（目标是 2-5 句话）。

当前摘要：
{summary}

新消息：
{messages}

只输出更新后的摘要，不要输出其他内容。"""




现在我们定义 `SummaryMemory` 类。构造函数设置两个存储区域：一个 `summary` 字符串（初始为空）和一个 `buffer` 列表用于最近消息。
- `_summarize` 方法调用 LLM 将当前摘要加上缓冲区压缩为一个更短的新摘要。
- `chat` 方法是魔法发生的地方。它构建一个包含运行摘要的系统提示词，将缓冲区发送给 LLM，并检查缓冲区是否已超过 `max_buffer_size`。如果是，则触发摘要周期并清空缓冲区。
- 最后，我们添加检查和工具方法。`get_context_snapshot` 精确显示 LLM 在下一次调用时会看到什么：摘要文本加上当前缓冲区内容。

In [12]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class SummaryMemory:
    """滚动摘要记忆，通过 LLM 压缩较早的对话轮次。"""

    def __init__(
        self,
        model,
        max_buffer_size: int = 6,
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.max_buffer_size = max_buffer_size

        # 状态
        self.summary: str = ""
        self.buffer: list[dict] = []

        # 追踪
        self.full_history: list[dict] = []
        self.summary_history: list[str] = []  # 每个摘要版本
        self.turn_token_usage: list[dict] = []
        self._summary_calls = 0

    # ── 摘要 ────────────────────────────────────────────
    def _format_messages_for_summary(self, messages: list[dict]) -> str:
        lines = []
        for msg in messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            lines.append(f"{role}: {msg['content']}")
        return "\n".join(lines)

    def _summarize(self) -> str:
        """将当前摘要 + 缓冲区压缩为一个新摘要。"""
        messages_text = self._format_messages_for_summary(self.buffer)
        prompt = SUMMARIZER_PROMPT.format(
            summary=self.summary or "(无先前摘要)",
            messages=messages_text,
        )
        response = self.model.invoke([HumanMessage(content=prompt)])
        self._summary_calls += 1
        return response.content.strip()
    # ── 聊天 ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        user_msg = {"role": "user", "content": user_input}
        self.buffer.append(user_msg)
        self.full_history.append(user_msg)

        # 构建包含摘要上下文的系统提示词
        system_text = self.system_prompt or ""
        if self.summary:
            if system_text:
                system_text += f"\n\n当前对话摘要：\n{self.summary}"
            else:
                system_text = f"当前对话摘要：\n{self.summary}"

        messages = []
        if system_text:
            messages.append(SystemMessage(content=system_text))
        for msg in self.buffer:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            else:
                messages.append(AIMessage(content=msg["content"]))

        response = self.model.invoke(messages)
        assistant_text = response.content

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.buffer.append(assistant_msg)
        self.full_history.append(assistant_msg)

        # 追踪 token 使用
        usage = {}
        if hasattr(response, 'usage_metadata'):
            usage = response.usage_metadata
        elif hasattr(response, 'response_metadata'):
            usage = response.response_metadata.get('token_usage', {})

        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
            "buffer_msgs": len(self.buffer),
            "summary_len": len(self.summary),
        })

        # 检查是否需要摘要
        if len(self.buffer) >= self.max_buffer_size:
            self.summary = self._summarize()
            self.summary_history.append(self.summary)
            self.buffer.clear()

        return assistant_text
    # ── 检查 ───────────────────────────────────────────────
    def get_context_snapshot(self) -> dict:
        """返回 LLM 当前看到的内容。"""
        return {
            "summary": self.summary,
            "buffer": list(self.buffer),
            "buffer_size": len(self.buffer),
            "total_messages": len(self.full_history),
            "summary_versions": len(self.summary_history),
        }

    def clear(self) -> None:
        self.summary = ""
        self.buffer.clear()
        self.full_history.clear()
        self.summary_history.clear()
        self.turn_token_usage.clear()
        self._summary_calls = 0

    def __repr__(self) -> str:
        return (
            f"SummaryMemory(buffer={len(self.buffer)}/{self.max_buffer_size}, "
            f"summary_versions={len(self.summary_history)}, "
            f"total_msgs={len(self.full_history)})"
        )


print("✓ SummaryMemory 类已定义")

✓ SummaryMemory 类已定义


## 使用示例：观察摘要演化

我们使用一个小缓冲区（`max_buffer_size=4`，意味着 2 轮触发一次摘要）。我们将植入几个事实，观察它们如何被压缩到摘要中。

In [13]:
mem = SummaryMemory(
    model=model,
    max_buffer_size=4,  # 每 2 轮触发一次摘要
    system_prompt="你是一个简洁的助手。用 1-2 句话回复。",
)

conversation = [
    "我叫 Carlos，来自布宜诺斯艾利斯。",
    "我是一名研究珊瑚礁的海洋生物学家。",
    "我最喜欢的编程语言是 Rust。",
    "目前你对我了解多少？",
]

for msg in conversation:
    print(f"👤 用户:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    snap = mem.get_context_snapshot()
    print(f"   📊 缓冲区: {snap['buffer_size']}/{mem.max_buffer_size} | 摘要版本: {snap['summary_versions']}")
    if snap["summary"]:
        print(f"   📝 当前摘要: {snap['summary'][:120]}...")
    print()

👤 用户:  我叫 Carlos，来自布宜诺斯艾利斯。
🤖 Agent: 你好 Carlos！布宜诺斯艾利斯是座美丽的城市。
   📊 缓冲区: 2/4 | 摘要版本: 0

👤 用户:  我是一名研究珊瑚礁的海洋生物学家。
🤖 Agent: 原来你是海洋生物学家！保护珊瑚礁的工作至关重要，尤其在气候变化的挑战下。
   📊 缓冲区: 0/4 | 摘要版本: 1
   📝 当前摘要: Carlos来自布宜诺斯艾利斯，是一名研究珊瑚礁的海洋生物学家。目前未提及未解决的问题或待定话题。...

👤 用户:  我最喜欢的编程语言是 Rust。
🤖 Agent: 很有趣！作为一名海洋生物学家，你竟然对系统级编程语言Rust有偏好。
   📊 缓冲区: 2/4 | 摘要版本: 1
   📝 当前摘要: Carlos来自布宜诺斯艾利斯，是一名研究珊瑚礁的海洋生物学家。目前未提及未解决的问题或待定话题。...

👤 用户:  目前你对我了解多少？
🤖 Agent: 我知道你是一名来自布宜诺斯艾利斯的海洋生物学家，研究珊瑚礁，并且偏爱Rust编程语言。
   📊 缓冲区: 0/4 | 摘要版本: 2
   📝 当前摘要: Carlos来自布宜诺斯艾利斯，是一名研究珊瑚礁的海洋生物学家，并且偏爱Rust编程语言。目前未提及未解决的问题或待定话题。...



让我们追踪摘要如何随时间演化。每个版本显示摘要器在合并一批消息后产生的内容。我们还打印当前缓冲区内容。

In [14]:
print("=== 摘要演化 ===\n")
for i, s in enumerate(mem.summary_history):
    print(f"版本 {i+1}:")
    print(f"  {s}")
    print()

print(f"=== 当前缓冲区（{len(mem.buffer)} 条消息）===")
for msg in mem.buffer:
    role = "用户" if msg["role"] == "user" else "助手"
    print(f"  {role}: {msg['content'][:80]}")

=== 摘要演化 ===

版本 1:
  Carlos来自布宜诺斯艾利斯，是一名研究珊瑚礁的海洋生物学家。目前未提及未解决的问题或待定话题。

版本 2:
  Carlos来自布宜诺斯艾利斯，是一名研究珊瑚礁的海洋生物学家，并且偏爱Rust编程语言。目前未提及未解决的问题或待定话题。

=== 当前缓冲区（0 条消息）===


## 关于摘要漂移

理论上，每轮摘要都是有损压缩，经过多次迭代后信息会逐步丢失——这叫**摘要漂移**（summary drift）。但在实际测试中，我们使用的 deepseek-chat 模型摘要能力过强，在小规模实验中几乎看不到信息丢失。

这不代表漂移不存在，而是说明：**当底层模型足够强时，摘要漂移在小规模对话中几乎感知不到**。漂移在以下条件下才会真正显现：对话极长（50+ 轮）、模型较弱或摘要温度较高、信息密度极高、事实之间彼此相似或容易混淆。

### 缓解策略

尽管当前模型表现很好，以下策略在生产环境中仍然值得了解：

1. **更好的摘要提示词**：明确要保留什么——专有名词、数字、偏好、决策及理由
2. **结构化摘要**：使用键值对格式（如 `事实: [...], 决策: [...], 待解决: [...]`），减少 LLM 意外丢弃字段的可能
3. **更大的缓冲区**：`max_buffer_size=20` 比 `4` 触发更少的摘要周期，复合误差更小
4. **摘要 + 缓冲混合**：在摘要旁保留最后 *k* 条原文消息（详见下个 notebook [04 摘要缓冲记忆](../04_summary_buffer_memory/)）
5. **实体提取**：将关键实体存入独立存储，摘要旁注入，即使摘要漂移结构化事实也不会丢
6. **定期完全重新摘要**：不要总是增量合并，偶尔从更大块的原始历史重新摘要，减少链式复合误差

## 讨论与权衡

### 优势
- **无界对话**：与滑动窗口不同，摘要记忆可以处理任意长的对话而不会丢失所有旧上下文。
- **有界的 token 成本**：摘要很紧凑。每轮成本大致恒定（摘要 + 缓冲区）。
- **优雅降级**：信息是逐渐压缩的，而不是硬截断。重要主题往往比次要细节保留更久。
- **灵活的压缩**：摘要提示词控制保留什么。你可以针对你的领域进行调整。

### 劣势
- **摘要漂移**：重复压缩引入累积信息丢失。具体数字、名称和细节最易受损。
- **额外的 LLM 调用**：每个摘要周期消耗 token 并增加延迟。使用激进的缓冲区大小 4 时，你每 2 轮就要进行一次额外的 LLM 调用。
- **非确定性记忆**：同一对话的两次运行可能产生不同的摘要，导致不同的 Agent 行为。
- **难以调试**：当 Agent "遗忘"某事时，很难判断是摘要器丢弃了它还是聊天模型忽略了它。
- **无法精确回忆**：Agent 永远无法从摘要历史中逐字引用你。

### 何时使用摘要记忆

| 场景 | 建议 |
|----------|---------------|
| 长对话（50+ 轮） | 很合适。在保留大意同时控制成本。 |
| 需要精确回忆早期事实 | 不理想。改用缓冲或检索增强记忆。 |
| 成本敏感型应用 | 很合适。比全缓冲便宜得多。 |
| 短对话（10 轮以下） | 开销不值得。使用缓冲或窗口。 |
| 多会话 Agent | 很合适。摘要可以轻松在会话之间传递。 |

### 成本模型
对于 *n* 轮对话，缓冲区大小为 *b*：
- **聊天调用：** *n*（与任何方案相同）
- **摘要调用：** floor(*n* / (*b*/2))（每次缓冲区刷新一次）
- **总额外成本：** 与 *n/b* 成正比。当 *b* 合理时，这只占总成本的一小部分。

## 进一步阅读

- [LangChain ConversationSummaryMemory](https://python.langchain.com/docs/modules/memory/types/summary?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：带有可自定义提示词的生产级实现
- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：将摘要与 token 限制缓冲区结合的混合方案
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：官方多轮对话模式
- [Recursive Summarization with LLMs](https://arxiv.org/abs/2301.13848)：关于迭代摘要质量的研究
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：包含记忆架构的概述

---

*← 上一章：02 滑动窗口记忆 · 下一章：[04 摘要缓冲记忆](../04_summary_buffer_memory/) →*

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：摘要提示词工程
编写三个不同的 `SUMMARIZER_PROMPT` 变体：一个优先保留事实，一个优先保留用户偏好，一个优先保留行动项。对每个变体运行相同的 10 轮对话，并排比较生成的摘要。

### 挑战 2：制造摘要漂移
尝试在以下条件下复现漂移：(1) 使用较弱的模型（如本地小模型），(2) 将 max_buffer_size 设为 2 并运行 50+ 轮，(3) 在每个填充轮中同时植入多个相似事实。观察事实何时开始丢失。

### 挑战 3：双层摘要
维护两个摘要：一个详细摘要（最近 10 轮）和一个高层摘要（之前的所有内容）。当详细摘要超出阈值时，将其压缩到高层摘要中。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--03-summary-memory--summary-memory)
